# Step 6: 모델 배포 & 영상 데모

## 학습 목표
이 노트북을 완료하면 다음을 이해할 수 있습니다:
- **SageMaker Endpoint** 배포 과정
- **숏폼 영상 분석** 파이프라인 (프레임 추출 → CNN 분석 → 결과 종합)
- **Gradio**를 활용한 영상/이미지 데모 UI 구축

## 영상 딥페이크 탐지 파이프라인

```
┌─────────────────────────────────────────────────────────────┐
│              숏폼 영상 딥페이크 탐지 파이프라인                │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  [사용자]                                                    │
│     │                                                       │
│     │ 영상/이미지 업로드                                      │
│     ▼                                                       │
│  ┌─────────────┐      ┌─────────────┐      ┌─────────────┐  │
│  │   Gradio    │ ──▶  │   프레임    │ ──▶  │  SageMaker  │  │
│  │   Web UI    │      │   추출      │      │  Endpoint   │  │
│  └─────────────┘      └─────────────┘      └─────────────┘  │
│                              │                    │         │
│                         3fps │              각 프레임        │
│                       샘플링  │              분석            │
│                              ▼                    │         │
│                       ┌─────────────┐             │         │
│                       │  다수결     │◀────────────┘         │
│                       │  투표      │                        │
│                       └─────────────┘                       │
│                              │                              │
│                              ▼                              │
│                       ┌─────────────┐                       │
│                       │ REAL / FAKE │                       │
│                       │  + 상세정보  │                       │
│                       └─────────────┘                       │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

## Endpoint 비용

| 인스턴스 | 시간당 비용 | 비고 |
|---------|-----------|------|
| ml.g4dn.xlarge | ~$0.74 | GPU (추론용) |

> ⚠️ **중요**: 실습 후 반드시 Endpoint를 삭제하세요!

In [ ]:
import json
import os
import sagemaker
from sagemaker.pytorch import PyTorchModel
from datetime import datetime
from pathlib import Path

# ============================================
# 프로젝트 경로 자동 설정
# ============================================
home_dir = Path.home()
PROJECT_ROOT = home_dir / 'deepfake-detection-sagemaker'
notebook_dir = PROJECT_ROOT / '6_demo'
os.chdir(notebook_dir)

print(f"Project Root: {PROJECT_ROOT}")
print(f"Current Dir: {os.getcwd()}")

# 설정 로드
config_path = PROJECT_ROOT / 'config.json'
with open(config_path, 'r') as f:
    config = json.load(f)

sagemaker_session = sagemaker.Session()
role = config['role']

# 최고 성능 기법의 모델 사용
best_method = config.get('best_method', 'full')
training_results = config.get('training_results', {})

if training_results and best_method in training_results:
    model_data = training_results[best_method]['model_data']
else:
    model_data = config['model_data']

# 고유한 Endpoint 이름 생성 (충돌 방지)
ENDPOINT_NAME = f"deepfake-{best_method}-{datetime.now().strftime('%Y%m%d-%H%M%S')}"

print(f"배포할 모델: {best_method.upper()} Fine-tuned")
print(f"모델 경로: {model_data}")
print(f"Endpoint Name: {ENDPOINT_NAME}")

## 6.1 SageMaker Endpoint 배포

### 배포 과정 이해

1. **PyTorchModel 생성**: S3의 model.tar.gz와 inference.py 지정
2. **deploy() 호출**: SageMaker가 자동으로:
   - EC2 인스턴스 프로비저닝 (ml.g4dn.xlarge)
   - Docker 컨테이너 시작
   - 모델 로드 및 웜업
3. **Endpoint 생성**: HTTPS 엔드포인트 URL 제공

### inference.py 역할

```python
def model_fn(model_dir):     # 모델 로드
def input_fn(data, type):    # 입력 전처리
def predict_fn(data, model): # 추론 실행
def output_fn(pred, type):   # 출력 후처리
```

> ⏱️ 배포에 약 5-10분 소요됩니다.

In [ ]:
# ============================================
# 🚀 SageMaker Endpoint 배포
# ============================================

pytorch_model = PyTorchModel(
    model_data=model_data,  # 최고 성능 기법의 모델
    role=role,
    entry_point='inference.py',
    source_dir='.',
    framework_version='2.0.0',
    py_version='py310'
)

print(f"배포할 모델: {best_method.upper()} Fine-tuned")
print("Endpoint 배포 중... (약 5-10분 소요)")

predictor = pytorch_model.deploy(
    initial_instance_count=1,
    instance_type='ml.g4dn.xlarge',
    endpoint_name=ENDPOINT_NAME
)

print(f"\n✅ Endpoint 배포 완료!")
print(f"   Endpoint 이름: {predictor.endpoint_name}")
print(f"   인스턴스: ml.g4dn.xlarge (NVIDIA T4 GPU)")

# config에 endpoint 이름 저장
config['endpoint_name'] = ENDPOINT_NAME
config_path = PROJECT_ROOT / 'config.json'
with open(config_path, 'w') as f:
    json.dump(config, f, indent=2)
print(f"\n💾 Endpoint 이름이 config.json에 저장되었습니다.")

## 6.2 Gradio 데모 실행

### Gradio란?
- ML 모델을 위한 **웹 UI 프레임워크**
- Python 코드 몇 줄로 데모 인터페이스 생성
- `share=True`로 외부 공유 가능한 URL 생성

### 데모 흐름

```
사용자가 이미지 업로드
      ↓
이미지 → JPEG 바이트로 변환
      ↓
SageMaker Endpoint 호출
      ↓
결과 (REAL/FAKE + 확신도) 표시
```

> 💡 `share=True`를 사용하면 임시 공개 URL이 생성되어 다른 사람과 공유할 수 있습니다.

In [ ]:
# Gradio & OpenCV 설치 (처음 한 번만 실행)
!pip install -q gradio opencv-python-headless
print("✅ Gradio & OpenCV 설치 완료!")

In [ ]:
# ============================================
# 🎬 숏폼 영상 딥페이크 탐지 데모
# ============================================
import gradio as gr
import boto3
import cv2
import numpy as np
from io import BytesIO
from PIL import Image
import tempfile
import os

runtime = boto3.client('sagemaker-runtime')

def analyze_single_frame(image):
    """단일 프레임 분석"""
    buffered = BytesIO()
    if isinstance(image, np.ndarray):
        image = Image.fromarray(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    image.save(buffered, format="JPEG")
    img_bytes = buffered.getvalue()
    
    response = runtime.invoke_endpoint(
        EndpointName=ENDPOINT_NAME,
        ContentType='application/x-image',
        Body=img_bytes
    )
    
    result = json.loads(response['Body'].read().decode())
    return result.get('prediction', 'Unknown'), result.get('confidence', 0)

def extract_frames(video_path, fps=3):
    """영상에서 프레임 추출 (3fps 기본)"""
    cap = cv2.VideoCapture(video_path)
    video_fps = cap.get(cv2.CAP_PROP_FPS)
    frame_interval = int(video_fps / fps) if video_fps > fps else 1
    
    frames = []
    frame_count = 0
    
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        if frame_count % frame_interval == 0:
            frames.append(frame)
        frame_count += 1
    
    cap.release()
    return frames

def detect_deepfake_video(video):
    """영상 딥페이크 탐지"""
    if video is None:
        return "영상을 업로드해주세요."
    
    # 프레임 추출
    frames = extract_frames(video, fps=3)
    
    if len(frames) == 0:
        return "영상에서 프레임을 추출할 수 없습니다."
    
    # 각 프레임 분석
    results = []
    fake_count = 0
    total_confidence = 0
    
    for i, frame in enumerate(frames):
        prediction, confidence = analyze_single_frame(frame)
        results.append((prediction, confidence))
        if prediction == 'FAKE':
            fake_count += 1
        total_confidence += confidence
    
    # 다수결 투표
    fake_ratio = fake_count / len(frames)
    avg_confidence = total_confidence / len(frames)
    
    final_prediction = "FAKE" if fake_ratio > 0.5 else "REAL"
    
    # 결과 포맷팅
    result_text = f"📊 분석 결과\n"
    result_text += f"{'='*40}\n\n"
    
    if final_prediction == "FAKE":
        result_text += f"🚨 **FAKE 영상 탐지!**\n\n"
    else:
        result_text += f"✅ **REAL 영상**\n\n"
    
    result_text += f"📈 상세 정보:\n"
    result_text += f"  - 분석 프레임 수: {len(frames)}개\n"
    result_text += f"  - FAKE 판정 프레임: {fake_count}개 ({fake_ratio:.1%})\n"
    result_text += f"  - REAL 판정 프레임: {len(frames)-fake_count}개 ({1-fake_ratio:.1%})\n"
    result_text += f"  - 평균 확신도: {avg_confidence:.1%}\n"
    
    return result_text

def detect_deepfake_image(image):
    """이미지 딥페이크 탐지"""
    if image is None:
        return "이미지를 업로드해주세요."
    
    prediction, confidence = analyze_single_frame(image)
    
    if prediction == 'FAKE':
        return f"🚨 FAKE 탐지!\n확신도: {confidence:.1%}"
    else:
        return f"✅ REAL\n확신도: {confidence:.1%}"

# Gradio 인터페이스 (탭으로 영상/이미지 분리)
with gr.Blocks(title="🎭 숏폼 딥페이크 영상 판별") as demo:
    gr.Markdown("# 🎭 숏폼 딥페이크 영상 판별 데모")
    gr.Markdown("영상 또는 이미지를 업로드하면 딥페이크 여부를 판별합니다. (KoDF Fine-tuned 모델 사용)")
    
    with gr.Tabs():
        with gr.TabItem("🎬 영상 분석"):
            gr.Markdown("### 숏폼 영상 업로드 (MP4, AVI 등)")
            gr.Markdown("영상에서 3fps로 프레임을 추출하여 각 프레임을 분석한 후 다수결로 최종 판정합니다.")
            with gr.Row():
                video_input = gr.Video(label="영상 업로드")
                video_output = gr.Textbox(label="탐지 결과", lines=10)
            video_btn = gr.Button("🔍 영상 분석", variant="primary")
            video_btn.click(fn=detect_deepfake_video, inputs=video_input, outputs=video_output)
        
        with gr.TabItem("🖼️ 이미지 분석"):
            gr.Markdown("### 단일 이미지 분석")
            with gr.Row():
                image_input = gr.Image(type="pil", label="이미지 업로드")
                image_output = gr.Textbox(label="탐지 결과", lines=5)
            image_btn = gr.Button("🔍 이미지 분석", variant="primary")
            image_btn.click(fn=detect_deepfake_image, inputs=image_input, outputs=image_output)

# 노트북에서 실행
demo.launch(share=True)

## 6.3 리소스 정리 (중요!)

### 비용 절감을 위해 반드시 실행하세요

Endpoint는 **실행 중인 동안 계속 과금**됩니다.
실습이 끝나면 아래 셀의 주석을 해제하고 실행하여 Endpoint를 삭제하세요.

| 리소스 | 시간당 비용 | 삭제 후 |
|--------|-----------|--------|
| ml.g4dn.xlarge Endpoint | ~$0.74 | $0 |

> ⚠️ **Workshop 종료 시 반드시 Endpoint를 삭제**하세요!

In [ ]:
# ⚠️ 실습 완료 후 반드시 실행하세요! (비용 절감)
# 아래 주석을 해제하고 실행하면 Endpoint가 삭제됩니다.

# predictor.delete_endpoint()
# print(f"✅ Endpoint '{ENDPOINT_NAME}' 삭제 완료!")
# print("더 이상 비용이 발생하지 않습니다.")

## 🎉 Workshop 완료!

축하합니다! **숏폼 딥페이크 영상 판별** 모델 Fine-tuning Workshop을 완료했습니다.

### 학습한 내용 요약

| 단계 | 내용 |
|------|------|
| **1. 데이터 준비** | S3에서 데이터 다운로드, 구조 이해 |
| **2. Before 평가** | Pretrained 모델의 한계 확인 (~70%) |
| **3. Fine-tuning** | Full, Freeze, LoRA 세 가지 기법 비교 |
| **4. After 평가** | Fine-tuned 모델 성능 확인 (~90%) |
| **5. 결과 비교** | 기법별 장단점 분석 |
| **6. 영상 데모** | SageMaker Endpoint + Gradio 영상 분석 |

### 영상 딥페이크 탐지 파이프라인

```
숏폼 영상 → 프레임 추출(3fps) → CNN 분석 → 다수결 투표 → REAL/FAKE
```

### 핵심 학습 포인트

1. **Domain Shift 문제**: Pretrained 모델은 다른 도메인에서 성능 저하
2. **Fine-tuning 효과**: 타겟 도메인 데이터로 학습 시 성능 크게 향상
3. **프레임 기반 분석**: 숏폼 영상에 효과적이고 빠름
4. **SageMaker 활용**: Experiments, Model Registry, Spot Instance

### 프로덕션 적용 시 고려사항

| 항목 | 워크샵 (데모) | 프로덕션 |
|------|-------------|----------|
| 추론 방식 | 실시간 Endpoint | 비동기 Inference |
| 스케일링 | 고정 1대 | Auto Scaling, Scale to Zero |
| 콜백 | 즉시 응답 | SNS 알림 |

### 다음 단계 (심화)

- **시공간 모델**: ViViT, TimeSformer로 시간적 패턴 분석
- **데이터 플라이휠**: 유저 투표 데이터로 모델 재학습
- **비동기 추론**: Scale to Zero로 비용 최적화

감사합니다!